# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [9]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)
df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^\w\s]", "", regex=True)
        .str.replace(" ", "_"))

df['effective_to_date'] = pd.to_datetime(df['effective_to_date'], format='%m/%d/%y')

df_sorted = df.sort_values(['customer', 'effective_to_date'])
latest_campaign = df_sorted.groupby('customer').tail(1)
new_df = latest_campaign[(latest_campaign['response'] == 'Yes') &
                              (latest_campaign['total_claim_amount'] < 1000)]

new_df

,unnamed_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_open_complaints,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type
6598,6598,AA16582,Washington,24127.504020,Yes,Basic,Bachelor,2011-01-26,Medical Leave,M,...,0.0,2,Personal Auto,Personal L2,Offer1,Agent,511.200000,Four-Door Car,Medsize,A
3159,3159,AA35519,Arizona,8002.308333,Yes,Basic,College,2011-01-11,Unemployed,F,...,0.0,3,Personal Auto,Personal L2,Offer1,Agent,513.600000,SUV,Medsize,A
10422,10422,AA56476,Arizona,5595.389905,Yes,Basic,High School or Below,2011-02-05,Employed,F,...,NaN,3,Corporate Auto,Corporate L2,Offer2,Call Center,340.800000,Four-Door Car,Medsize,NaN
6360,6360,AB21519,California,3861.486269,Yes,Extended,College,2011-01-06,Employed,F,...,0.0,1,Personal Auto,Personal L3,Offer2,Branch,281.110788,Four-Door Car,Medsize,NaN
2834,2834,AB96670,California,2393.915369,Yes,Basic,College,2011-01-31,Unemployed,M,...,0.0,1,Personal Auto,Personal L1,Offer1,Branch,425.266308,Four-Door Car,Medsize,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10433,10433,ZW79814,California,6689.022728,Yes,Extended,High School or Below,2011-02-13,Employed,M,...,NaN,7,Personal Auto,Personal L1,Offer2,Agent,599.000883,Two-Door Car,Medsize,A
306,306,ZX73673,Arizona,5555.830205,Yes,Basic,Bachelor,2011-02-06,Employed,M,...,0.0,6,Personal Auto,Personal L3,Offer2,Call Center,471.443077,Four-Door Car,Small,NaN
9307,9307,ZX93551,Arizona,2867.312197,Yes,Extended,Bachelor,2011-01-03,Retired,F,...,0.0,1,Personal Auto,Personal L3,Offer2,Call Center,374.400000,Four-Door Car,Medsize,NaN
8615,8615,ZZ22047,Washington,5548.031892,Yes,Basic,Bachelor,2011-01-13,Employed,M,...,0.0,9,Corporate Auto,Corporate L3,Offer1,Agent,331.200000,Four-Door Car,Medsize,NaN


2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [16]:
yes_customers = df[df['response'] == 'Yes']
grouped = yes_customers.groupby(['policy_type', 'gender']).agg({
    'customer_lifetime_value': 'mean',
    'total_claim_amount': 'mean'
}).reset_index()
grouped

# High CLV + low average claims = ideal, low-risk, profitable segment → typically Corporate Auto for both genders.
# Lower CLV + high claims = riskier and less profitable → usually Personal Auto.
# Gender differences can indicate which segment is slightly more profitable or has lower claim risk.

,policy_type,gender,customer_lifetime_value,total_claim_amount
0,Corporate Auto,F,7712.628736,433.738499
1,Corporate Auto,M,7944.465414,408.582459
2,Personal Auto,F,8339.791842,452.965929
3,Personal Auto,M,7448.383281,457.010178
4,Special Auto,F,7691.584111,453.280164
5,Special Auto,M,8247.088702,429.527942


3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [17]:
state_counts = df['state'].value_counts().reset_index()
state_counts.columns = ['state', 'num_customers']

high_states = state_counts[state_counts['num_customers'] > 500]

high_states

,state,num_customers
0,California,3552
1,Oregon,2909
2,Arizona,1937
3,Nevada,993
4,Washington,888


4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [18]:
clv_stats = df.groupby(['education', 'gender'])['customer_lifetime_value'].agg(['max', 'min', 'median']).reset_index()

clv_stats
# Higher education correlates with higher customer lifetime value.
# Gender differences exist but are minor.
# Customers with higher education are likely more profitable, and retention efforts for these groups could yield higher returns.

,education,gender,max,min,median
0,Bachelor,F,73225.95652,1904.000852,5640.505303
1,Bachelor,M,67907.27050,1898.007675,5548.031892
2,College,F,61850.18803,1898.683686,5623.611187
3,College,M,61134.68307,1918.119700,6005.847375
4,Doctor,F,44856.11397,2395.570000,5332.462694
5,Doctor,M,32677.34284,2267.604038,5577.669457
6,High School or Below,F,55277.44589,2144.921535,6039.553187
7,High School or Below,M,83325.38119,1940.981221,6286.731006
8,Master,F,51016.06704,2417.777032,5729.855012
9,Master,M,50568.25912,2272.307310,5579.099207


## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

In [19]:
df['effective_to_date'] = pd.to_datetime(df['effective_to_date'], format='%m/%d/%y')
df['month'] = df['effective_to_date'].dt.month
policies_by_state_month = pd.pivot_table(
    df,
    index='state',
    columns='month',
    values='policy',      
    aggfunc='count',      
    fill_value=0          
).sort_index(axis=1)
policies_by_state_month

month,1,2
state,,
Arizona,1008,929
California,1918,1634
Nevada,551,442
Oregon,1565,1344
Washington,463,425


6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

In [20]:
df['effective_to_date'] = pd.to_datetime(df['effective_to_date'], format='%m/%d/%y')

df['month'] = df['effective_to_date'].dt.month

state_month_counts = df.groupby(['state', 'month'])['policy'].count().reset_index(name='num_policies')

state_totals = state_month_counts.groupby('state')['num_policies'].sum().reset_index()
state_totals = state_totals.sort_values(by='num_policies', ascending=False)

top_3_states = state_totals.head(3)['state'].tolist()

top_states_data = state_month_counts[state_month_counts['state'].isin(top_3_states)]

top_states_pivot = top_states_data.pivot_table(
    index='state',
    columns='month',
    values='num_policies',
    fill_value=0
).sort_index(axis=1)

top_states_pivot

month,1,2
state,,
Arizona,1008.0,929.0
California,1918.0,1634.0
Oregon,1565.0,1344.0


7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [21]:
# your code goes here
channel_response = df.groupby('sales_channel')['response'].apply(lambda x: (x == 'Yes').mean()).reset_index()

channel_response.rename(columns={'response': 'response_rate'}, inplace=True)

channel_response = channel_response.sort_values(by='response_rate', ascending=False)

channel_response

,sales_channel,response_rate
0,Agent,0.180053
3,Web,0.108856
1,Branch,0.107876
2,Call Center,0.103223
